In [1]:
from src.LM.lm_roberta_xml import (
    tokenizer, model,
    create_dataloaders, train_model, evaluate_model, predict_sentence
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from src.data_loader import load_data
from src.preprocess import clean_text

import torch
import matplotlib.pyplot as plt
import numpy as np


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:

df = load_data("/home/xkubanova_126831/bakalarka/e_com_dataset/data/ecommerceDataset.csv")
df['Text'] = df['Text'].apply(clean_text)  

le = LabelEncoder()
df['Category_encoded'] = le.fit_transform(df['Category'])

# ulozenie katagorii
target_names = le.classes_.tolist()


# rozdelenie datasetu
X_train_text, X_test_text, y_train_encoded, y_test_encoded = train_test_split(
    df['Text'].tolist(), 
    df['Category_encoded'].tolist(),
    test_size=0.2, 
    random_state=42,
    stratify=df['Category_encoded']
)


Dataset shape: (50425, 2)

Categories samples:

Books:
  Inner Engineering: A Yogi's Guide to Joy About the Author Sadhguru Jaggi VasudevSADHGURU is a yogi, mystic, and visionary who established the Isha Foundation, a nonprofit dedicated to the cultivation of human potential. He belongs to no particular tradition, and his scientific methods for self-transformation have universal appeal. Sadhguru has been an in?uential voice at global forums including th...

Clothing & Accessories:
  Woopower 36M Pink for 024M Baby Trouser Top Sets3Pcs Boy Girl Hooded Topsstriped Pantshairband Outfits36Mpink Size name36m colourpink description100 brand new and type children setgender unisexfor season autumn springcoloroptional pink greensize table 7080 90 100cminchsize tops length bust pants length age70 32 1260 46 1811 37 1457 36m80 34 1339 48 1890 39 1535 612m90 36 1417 52 2047 41 1614 1218m...

Electronics:
  Dell 19.5V-3.34AMP 65W Laptop Adapter (Without power Cord) Design Features of Dell Laptop - 

In [3]:
# konverzia textu a labelov na PyTorch Dataloaders objekty
# batchovanie, rozdelenie na test / train
train_loader, test_loader = create_dataloaders(X_train_text, y_train_encoded, X_test_text, y_test_encoded)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



# trenovanie LM
trained_model = train_model(model, train_loader, epochs=2)
evaluate_model(trained_model, test_loader, label_names=target_names)
model.eval()



Epoch 1/2, Loss: 0.2448
Epoch 2/2, Loss: 0.1820
                        precision    recall  f1-score   support

                 Books       0.98      0.95      0.96      2364
Clothing & Accessories       0.97      0.98      0.97      1734
           Electronics       0.96      0.96      0.96      2124
             Household       0.96      0.98      0.97      3863

              accuracy                           0.97     10085
             macro avg       0.97      0.96      0.97     10085
          weighted avg       0.97      0.97      0.97     10085



XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [4]:
from src.LM.lm_roberta_xml import predict_sentence
sentence = "Artography Studio – Serene Bloom Premium Floral Canvas Wall Art (Set of 2, 12 x 18 inch) – MulticolorTransform your home décor with this elegant floral art set that brings a touch of tranquility to any room. These premium canvas prints feature vibrant high-definition artwork of blooming flowers, reproduced using cutting-edge Epson and HP printing technology for long-lasting brilliance. Each canvas is printed on thick, museum-quality material with UV-resistant inks that ensure colors remain vivid for years."
predicted_category = predict_sentence(sentence, trained_model, tokenizer, target_names, device="cuda")
print(f"Predicted category: {predicted_category}")


Predicted category: Household


In [9]:
def get_attentions_and_tokens(model, tokenizer, sentence: str, device="cuda"):
    # tokenize
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=False
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # --- temporarily switch to eager attention + enable output_attentions ---
    old_impl = getattr(model.config, "attn_implementation", None)
    old_output_attn = getattr(model.config, "output_attentions", None)

    # set to eager mode so attentions are supported
    model.config.attn_implementation = "eager"
    model.config.output_attentions = True

    with torch.no_grad():
        outputs = model(**inputs)   # 👈 no attn_implementation kwarg here

    # restore original config (optional but nice)
    if old_impl is not None:
        model.config.attn_implementation = old_impl
    if old_output_attn is not None:
        model.config.output_attentions = old_output_attn

    # outputs.attentions: tuple[num_layers] of [1, num_heads, seq_len, seq_len]
    attentions = outputs.attentions
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    return attentions, tokens


In [10]:
import matplotlib.pyplot as plt
import numpy as np

def plot_attention_head(attentions, tokens, layer=0, head=0):
    attn = attentions[layer][0, head].detach().cpu().numpy()
    seq_len = len(tokens)

    fig, ax = plt.subplots(figsize=(max(6, seq_len * 0.4), max(6, seq_len * 0.4)))
    im = ax.imshow(attn)

    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticklabels(tokens)

    ax.set_xlabel("Attended token")
    ax.set_ylabel("Query token")
    ax.set_title(f"Attention heatmap – layer {layer}, head {head}")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


In [11]:
def plot_attention_heads_grid(attentions, tokens, layer=0, max_heads=8):
    attn_layer = attentions[layer][0]
    num_heads = min(attn_layer.shape[0], max_heads)

    fig, axes = plt.subplots(1, num_heads, figsize=(num_heads * 3, 3))
    if num_heads == 1:
        axes = [axes]

    for h in range(num_heads):
        ax = axes[h]
        ax.imshow(attn_layer[h].detach().cpu().numpy())
        ax.set_title(f"Head {h}")
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(f"Layer {layer} — attention heads")
    plt.tight_layout()
    plt.show()


In [12]:
short_sentence = (
    "Serene Bloom floral canvas wall art set that decorates the living room."
)

attentions, tokens = get_attentions_and_tokens(
    trained_model,
    tokenizer,
    short_sentence,   # 👈 shorter version for nicer figure
    device=device
)

# Single head for thesis figure
plot_attention_head(attentions, tokens, layer=6, head=3)

# Optional: grid of multiple heads
plot_attention_heads_grid(attentions, tokens, layer=6, max_heads=8)

ValueError: The `output_attentions` attribute is not supported when using the `attn_implementation` set to sdpa. Please set it to 'eager' instead.